# 02 — KPI Calculations
**Project:** Supply Chain Intelligence System  
**Input:** `data/processed/master_orders.csv`  
**Goal:** Calculate all supply chain KPIs and export dashboard-ready CSVs for Tableau.

---
## KPIs Calculated
1. Fulfillment latency (p50, p95) — overall, by state, by category
2. On-time delivery rate — overall, by seller, by region
3. Seller scorecard — ranking + risk flags
4. Stockout proxy — demand gaps by category
5. Monthly KPI trends — for time series in dashboard

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings_import = __import__('warnings')
warnings_import.filterwarnings('ignore')

master = pd.read_csv('../data/processed/master_orders.csv', parse_dates=['order_purchase_timestamp'])
print(f'Loaded {len(master):,} rows')
master.head(2)

## KPI 1 — Fulfillment Latency (p50 & p95)

In [ ]:
# Overall latency
p50 = master['fulfillment_days'].median()
p95 = master['fulfillment_days'].quantile(0.95)
print(f'Overall Fulfillment Latency')
print(f'  p50 (median): {p50:.1f} days')
print(f'  p95:          {p95:.1f} days')
print(f'  (p95 = worst-case experience for 1 in 20 customers)')

# By customer state
latency_by_state = master.groupby('customer_state')['fulfillment_days'].agg(
    p50=lambda x: x.median(),
    p95=lambda x: x.quantile(0.95),
    order_count='count'
).reset_index().sort_values('p50', ascending=False)

print('\nTop 5 states with worst median latency:')
print(latency_by_state.head(5).to_string(index=False))

In [ ]:
# By product category
latency_by_cat = master.groupby('category_en')['fulfillment_days'].agg(
    p50=lambda x: x.median(),
    p95=lambda x: x.quantile(0.95),
    order_count='count'
).reset_index()

# Only include categories with meaningful volume
latency_by_cat = latency_by_cat[latency_by_cat['order_count'] >= 100]
latency_by_cat = latency_by_cat.sort_values('p50', ascending=False)

print('Top 10 categories with worst latency:')
print(latency_by_cat.head(10).to_string(index=False))

## KPI 2 — On-Time Delivery Rate

In [ ]:
overall_otr = master['on_time'].mean()
print(f'Overall On-Time Delivery Rate: {overall_otr*100:.1f}%')

# By state
otr_by_state = master.groupby('customer_state').agg(
    on_time_rate   = ('on_time', 'mean'),
    total_orders   = ('order_id', 'nunique'),
    avg_latency    = ('fulfillment_days', 'mean')
).reset_index()
otr_by_state['on_time_pct'] = (otr_by_state['on_time_rate'] * 100).round(1)
otr_by_state['flag_risk'] = otr_by_state['on_time_rate'] < 0.75  # Flag states below 75%

print(f'\nStates at risk (< 75% on-time): {otr_by_state["flag_risk"].sum()}')
print(otr_by_state[otr_by_state['flag_risk']].sort_values('on_time_pct').to_string(index=False))

## KPI 3 — Seller Scorecard

In [ ]:
seller_kpis = master.groupby('seller_id').agg(
    total_orders      = ('order_id', 'nunique'),
    on_time_rate      = ('on_time', 'mean'),
    avg_latency_days  = ('fulfillment_days', 'mean'),
    p95_latency       = ('fulfillment_days', lambda x: x.quantile(0.95)),
    avg_review_score  = ('review_score', 'mean'),
    total_revenue_brl = ('payment_value', 'sum'),
    seller_state      = ('seller_state', 'first')
).reset_index()

# Only score sellers with enough orders
seller_kpis = seller_kpis[seller_kpis['total_orders'] >= 10]

# Composite performance score (0-100)
# Weights: on-time 50%, review 30%, latency 20%
seller_kpis['latency_score'] = 100 - (
    (seller_kpis['avg_latency_days'] - seller_kpis['avg_latency_days'].min()) /
    (seller_kpis['avg_latency_days'].max() - seller_kpis['avg_latency_days'].min()) * 100
)
seller_kpis['review_score_norm'] = (seller_kpis['avg_review_score'] - 1) / 4 * 100
seller_kpis['performance_score'] = (
    seller_kpis['on_time_rate'] * 50 +
    seller_kpis['review_score_norm'] * 0.30 +
    seller_kpis['latency_score'] * 0.20
).round(1)

# Risk tier
seller_kpis['risk_tier'] = pd.cut(
    seller_kpis['on_time_rate'],
    bins=[0, 0.65, 0.80, 0.92, 1.0],
    labels=['High Risk', 'At Risk', 'Acceptable', 'Top Performer']
)

print('Seller risk distribution:')
print(seller_kpis['risk_tier'].value_counts())
print(f'\nTotal sellers scored: {len(seller_kpis)}')

## KPI 4 — Monthly Trends (Dashboard Time Series)

In [ ]:
master['year_month'] = master['order_purchase_timestamp'].dt.to_period('M').astype(str)

monthly_kpis = master.groupby('year_month').agg(
    total_orders     = ('order_id', 'nunique'),
    on_time_rate     = ('on_time', 'mean'),
    avg_latency_days = ('fulfillment_days', 'mean'),
    p95_latency      = ('fulfillment_days', lambda x: x.quantile(0.95)),
    avg_review       = ('review_score', 'mean'),
    total_revenue    = ('payment_value', 'sum')
).reset_index()

monthly_kpis['on_time_pct'] = (monthly_kpis['on_time_rate'] * 100).round(1)

print('Monthly KPI sample:')
print(monthly_kpis.tail(6).to_string(index=False))

## Export All KPI Tables for Tableau

In [ ]:
monthly_kpis.to_csv('../data/processed/kpi_monthly_trends.csv', index=False)
latency_by_state.to_csv('../data/processed/kpi_latency_by_state.csv', index=False)
latency_by_cat.to_csv('../data/processed/kpi_latency_by_category.csv', index=False)
otr_by_state.to_csv('../data/processed/kpi_ontime_by_state.csv', index=False)
seller_kpis.to_csv('../data/processed/kpi_seller_scorecard.csv', index=False)

print('Exported 5 KPI tables to data/processed/')
print('These files connect directly to Tableau dashboard.')